# AcousticAcre: maize and tomato on real field photos

**What this does:** measures the current crop model on real-scene maize and tomato photos it has never seen (PlantDoc), then retrains with some of those photos and measures again on the photos held back. You get honest **before** and **after** numbers, and a new crop pack (v0.2.0) for the app.

1. **Runtime → Change runtime type → T4 GPU → Save.** (If no GPU is offered, steps 1–3 still work; step 4, training, needs the GPU.)
2. Upload `acousticacre.zip` (made with `git archive`, see the instructions you were given) using the folder icon on the left.
3. Run the cells in order. Total time: about 60–90 minutes, mostly downloading and training.

Quote only the numbers the cells print, each with its source name.

In [ ]:
!unzip -q -o acousticacre.zip -d acousticacre
!pip -q install timm onnx onnxruntime importlib_resources datasets
import torch; print('GPU:', torch.cuda.is_available())

## 1. Updated scripts (adds PlantDoc field photos and the field evaluation)

In [ ]:
%%writefile /content/acousticacre/ml/train/crop_prepare.py
"""Download public leaf datasets via TensorFlow Datasets (preinstalled on Colab) and write
an image folder in our diagnosis-ID classes.

  cassava       field photos (Uganda), own train/validation/test splits  -> FIELD test set
  plant_village lab photos (plain background), random split              -> LAB test set (not field accuracy)
  plantdoc      web-collected real-scene photos (maize, tomato), deduplicated,
                stratified 55/15/30 split                                -> FIELD test set for maize/tomato

    python -m train.crop_prepare --out data/crop --max-per-class 1500

Output: data/crop/{train,val,test}/<diagnosis_id>/<source>_<n>.jpg  + data/crop/sources.json
"""
from __future__ import annotations

import argparse
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

CASSAVA = {"cbb": "cassava_bacterial_blight", "cbsd": "cassava_brown_streak", "cgm": "cassava_green_mite",
           "cmd": "cassava_mosaic", "healthy": "cassava_healthy"}


def map_plantvillage(name: str) -> str | None:
    """PlantVillage label name -> our diagnosis ID (None = not used)."""
    n = name.lower()
    if n.startswith("corn"):
        if "common_rust" in n:
            return "maize_common_rust"
        if "northern_leaf_blight" in n:
            return "maize_northern_leaf_blight"
        if "cercospora" in n or "gray_leaf_spot" in n:
            return "maize_gray_leaf_spot"
        if "healthy" in n:
            return "maize_healthy"
        return None
    if n.startswith("tomato"):
        if "early_blight" in n:
            return "tomato_early_blight"
        if "late_blight" in n:
            return "tomato_late_blight"
        if "yellow_leaf_curl" in n:
            return "tomato_yellow_leaf_curl"
        if "healthy" in n:
            return "tomato_healthy"
        return "tomato_other_leaf_problem"   # bacterial spot, leaf mold, septoria, mites, target spot, mosaic
    return None


PLANTDOC = {
    "Corn Gray leaf spot": "maize_gray_leaf_spot",
    "Corn leaf blight": "maize_northern_leaf_blight",
    "Corn rust leaf": "maize_common_rust",
    "Tomato Early blight leaf": "tomato_early_blight",
    "Tomato leaf": "tomato_healthy",
    "Tomato leaf late blight": "tomato_late_blight",
    "Tomato leaf yellow virus": "tomato_yellow_leaf_curl",
    "Tomato Septoria leaf spot": "tomato_other_leaf_problem",
    "Tomato leaf bacterial spot": "tomato_other_leaf_problem",
    "Tomato leaf mosaic virus": "tomato_other_leaf_problem",
    "Tomato mold leaf": "tomato_other_leaf_problem",
    "Tomato two spotted spider mites leaf": "tomato_other_leaf_problem",
}


PLANTDOC_NOTE = ("PlantDoc (Singh et al. 2019, CC BY-SA 4.0), web-collected real-scene maize and tomato photos, "
                 "near-duplicates removed before a stratified 55/15/30 split. Quotable as a real-scene result "
                 "for maize/tomato, with the caveat that the photos are not from Nigerian farms.")


def ahash(img, n=8):
    """Tiny perceptual hash: catches the same photo re-uploaded at another size."""
    g = img.convert("L").resize((n, n))
    px = list(g.getdata())
    m = sum(px) / len(px)
    return sum(1 << i for i, v in enumerate(px) if v > m) & 0xFFFFFFFFFFFFFFFF


def add_plantdoc(out, save, rnd, fractions=(0.55, 0.15, 0.30)):
    """PlantDoc (Singh et al. 2019, CC BY-SA 4.0) from the Hugging Face mirror.
    Duplicates are removed BEFORE splitting so one photo never lands in train and test."""
    from datasets import load_dataset
    import numpy as np
    from PIL import Image
    # PlantDoc was collected from the web, so some photos may be PlantVillage images.
    # Hash every maize/tomato lab photo already written and drop PlantDoc matches (no leakage).
    lab = []
    for f in Path(out).glob("*/*/plantvillage_lab_*.jpg"):
        if f.parent.name.startswith(("maize_", "tomato_")):
            with Image.open(f) as im:
                lab.append(ahash(im))
    print(f"plantdoc: checking against {len(lab)} lab photos for overlap")
    seen = np.array(lab, dtype=np.uint64)

    def near_dup(h, max_bits=3):  # reads the current 'seen' from the enclosing scope
        """True if h is within max_bits differing bits of any hash seen so far."""
        if seen.size == 0:
            return False
        x = np.bitwise_xor(seen, np.uint64(h)).view(np.uint8)
        return bool((np.unpackbits(x).reshape(-1, 64).sum(1) <= max_bits).any())
    ds = load_dataset("Project-AgML/plant_doc_classification", split="train")
    names = ds.features["label"].names
    by_class, dupes = defaultdict(list), 0
    for i, ex in enumerate(ds):
        diag = PLANTDOC.get(names[ex["label"]])
        if not diag:
            continue
        img = ex["image"].convert("RGB")
        h = ahash(img)
        if near_dup(h):
            dupes += 1
            continue
        seen = np.append(seen, np.uint64(h))
        by_class[diag].append((i, img))
    for diag, items in sorted(by_class.items()):
        rnd.shuffle(items)
        n = len(items)
        a, b = int(n * fractions[0]), int(n * (fractions[0] + fractions[1]))
        for split, part in (("train", items[:a]), ("val", items[a:b]), ("test", items[b:])):
            for i, img in part:
                save(split, diag, "plantdoc_field", np.asarray(img), i)
    print(f"plantdoc: {sum(len(v) for v in by_class.values())} images kept, {dupes} duplicates dropped")
    return dupes


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data/crop")
    ap.add_argument("--max-per-class", type=int, default=1500, help="per split-source-class cap (train)")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--skip-plantvillage", action="store_true")
    ap.add_argument("--skip-plantdoc", action="store_true")
    ap.add_argument("--only-plantdoc", action="store_true", help="add PlantDoc to an existing data folder")
    a = ap.parse_args(argv)

    import tensorflow_datasets as tfds
    from PIL import Image

    out = Path(a.out)
    rnd = random.Random(a.seed)
    counts: dict = defaultdict(Counter)
    caps = {"train": a.max_per_class, "val": max(150, a.max_per_class // 5), "test": max(150, a.max_per_class // 5)}

    def save(split, diag, source, img, i):
        if counts[(split, source)][diag] >= caps[split]:
            return
        d = out / split / diag
        d.mkdir(parents=True, exist_ok=True)
        im = Image.fromarray(img)
        im.thumbnail((384, 384))                       # keep files small; training crops to 224
        im.save(d / f"{source}_{i:06d}.jpg", quality=90)
        counts[(split, source)][diag] += 1

    if a.only_plantdoc:
        add_plantdoc(out, save, rnd)
        for s_ in ("train", "val", "test"):
            print("plantdoc", s_, dict(counts[(s_, "plantdoc_field")]))
        src = out / "sources.json"
        meta = json.loads(src.read_text()) if src.exists() else {}
        meta["plantdoc_field"] = PLANTDOC_NOTE
        meta.setdefault("counts", {}).update({f"{s_}/plantdoc_field": dict(counts[(s_, "plantdoc_field")]) for s_ in ("train", "val", "test")})
        src.write_text(json.dumps(meta, indent=2))
        return

    # --- cassava: field photos, official splits ---------------------------------
    ds, info = tfds.load("cassava", with_info=True)
    names = info.features["label"].names
    for tf_split, ours in [("train", "train"), ("validation", "val"), ("test", "test")]:
        for i, ex in enumerate(tfds.as_numpy(ds[tf_split])):
            diag = CASSAVA.get(names[int(ex["label"])])
            if diag:
                save(ours, diag, "cassava_field", ex["image"], i)
        print("cassava", tf_split, dict(counts[(ours, "cassava_field")]))

    # --- PlantVillage: lab photos, random 80/10/10 ------------------------------
    if not a.skip_plantvillage:
        ds, info = tfds.load("plant_village", with_info=True, split="train")
        names = info.features["label"].names
        for i, ex in enumerate(tfds.as_numpy(ds)):
            diag = map_plantvillage(names[int(ex["label"])])
            if not diag:
                continue
            r = rnd.random()
            split = "train" if r < 0.8 else "val" if r < 0.9 else "test"
            save(split, diag, "plantvillage_lab", ex["image"], i)
        for s in ("train", "val", "test"):
            print("plantvillage", s, dict(counts[(s, "plantvillage_lab")]))

    if not a.skip_plantdoc:
        add_plantdoc(out, save, rnd)
        for s_ in ("train", "val", "test"):
            print("plantdoc", s_, dict(counts[(s_, "plantdoc_field")]))

    (out / "sources.json").write_text(json.dumps({
        "cassava_field": "TFDS 'cassava' (iCassava 2019, field photos, Uganda); official splits",
        "plantvillage_lab": "TFDS 'plant_village' (lab photos, plain background); random image split. "
                            "Test accuracy here is NOT field accuracy.",
        "plantdoc_field": PLANTDOC_NOTE,
        "counts": {f"{s}/{src}": dict(c) for (s, src), c in counts.items()},
    }, indent=2))
    print("done ->", out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/acousticacre/ml/eval/field_eval.py
"""Score a crop pack (the files the app ships) on held-out photos of one source, the way the app
decides: temperature + thresholds from the manifest, and the farmer's chosen crop = the photo's
true crop, so a confident answer about ANOTHER crop counts as a "wrong crop" retake.

    python -m eval.field_eval --pack path/to/public/models/crop --data data/crop/test --source plantdoc_field
    python -m eval.field_eval --pack runs/crop --data data/crop/test --source plantdoc_field   (a new run)

Prints and writes <out>.json: accuracy, coverage, accuracy when it answers, how many wrong answers
were caught as "not sure", and per-class numbers. Only quote it with its source name.
"""
from __future__ import annotations

import argparse
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from eval.decision import decide, softmax  # noqa: E402


def load_pack(pack: Path):
    """A shipped pack has manifest.json; a fresh training run has pack_params.json."""
    if (pack / "manifest.json").exists():
        m = json.loads((pack / "manifest.json").read_text())
        model, T, th = m["model"], m["temperature"], m["thresholds"]
        mean, std = m["input"].get("mean", [0.5] * 3), m["input"].get("std", [0.5] * 3)
    else:
        m = json.loads((pack / "pack_params.json").read_text())
        model, T, th = m["model_file"], m["temperature"], m["thresholds"]
        mean, std = m["input"]["mean"], m["input"]["std"]
    labels = [l["label"] for l in json.loads((pack / "labels.json").read_text())]
    return pack / model, labels, float(T), th, mean, std


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--pack", required=True)
    ap.add_argument("--data", default="data/crop/test")
    ap.add_argument("--source", default="plantdoc_field")
    ap.add_argument("--out", default=None)
    a = ap.parse_args(argv)

    import onnxruntime as ort
    from PIL import Image

    model_path, labels, T, th, mean, std = load_pack(Path(a.pack))
    sess = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
    mean_, std_ = np.array(mean, np.float32)[:, None, None], np.array(std, np.float32)[:, None, None]

    def tf(im):
        """Same as training's Resize(256) + CenterCrop(224) + Normalize, in plain PIL/numpy."""
        w, h = im.size
        s = 256 / min(w, h)
        im = im.resize((max(224, round(w * s)), max(224, round(h * s))), Image.BILINEAR)
        W, H = im.size
        l, t = int(round((W - 224) / 2.0)), int(round((H - 224) / 2.0))
        x = np.asarray(im.crop((l, t, l + 224, t + 224)), np.float32).transpose(2, 0, 1) / 255.0
        return (x - mean_) / std_

    files = sorted(Path(a.data).glob(f"*/{a.source}_*.jpg"))
    if not files:
        sys.exit(f"No {a.source}_*.jpg images under {a.data}. Run crop_prepare first.")
    Z, Y = [], []
    for i in range(0, len(files), 32):
        batch = files[i:i + 32]
        x = np.stack([tf(Image.open(f).convert("RGB")) for f in batch]).astype(np.float32)
        Z.append(sess.run(["logits"], {"input": x})[0])
        Y += [labels.index(f.parent.name) for f in batch]
    Z, Y = np.concatenate(Z), np.array(Y)
    P = softmax(Z, T)
    d = decide(P, th)                       # class index, or -1 = "not sure"
    top = P.argmax(1)
    crop = lambda k: labels[k].split("_")[0]

    outcome = []
    for y, t, dk in zip(Y, top, d):
        if dk == -1:
            outcome.append("not_sure")
        elif crop(dk) != crop(y):
            outcome.append("wrong_crop_retake")   # app asks the farmer to re-pick; no diagnosis given
        elif dk == y:
            outcome.append("correct")
        else:
            outcome.append("wrong_diagnosis")
    oc = Counter(outcome)
    n = len(Y)
    answered = oc["correct"] + oc["wrong_diagnosis"]
    wrong_top1 = (top != Y)

    per_class = defaultdict(lambda: Counter())
    for y, o in zip(Y, outcome):
        per_class[labels[y]][o] += 1

    res = {
        "source": a.source, "pack": str(a.pack), "n_images": int(n),
        "top1_accuracy": float((top == Y).mean()),
        "outcomes": dict(oc),
        "coverage": answered / n,
        "accuracy_when_answering": (oc["correct"] / answered) if answered else None,
        "wrong_answers_caught": float(np.mean([o != "wrong_diagnosis" for o, w in zip(outcome, wrong_top1) if w])) if wrong_top1.any() else None,
        "per_class": {k: dict(v) for k, v in sorted(per_class.items())},
        "note": "Real-scene photos never used for training. The app's wrong-crop rule is applied.",
    }
    out = Path(a.out or f"field_eval_{a.source}.json")
    out.write_text(json.dumps(res, indent=2))

    print(f"\n{a.source}: {n} held-out photos")
    print(f"  top-1 accuracy (any answer)      {res['top1_accuracy']:.1%}")
    print(f"  app gives a diagnosis on          {res['coverage']:.1%}")
    if answered:
        print(f"  ...and is right on                {res['accuracy_when_answering']:.1%}")
    if res["wrong_answers_caught"] is not None:
        print(f"  wrong answers caught (not sure or wrong-crop retake) {res['wrong_answers_caught']:.1%}")
    print(f"  outcomes: {dict(oc)}")
    for k, v in res["per_class"].items():
        tot = sum(v.values())
        print(f"    {k:28s} {tot:4d} photos  correct {v.get('correct', 0) / tot:.0%}")
    print(f"saved {out}")


if __name__ == "__main__":
    main()


## 2. Download the photos
Cassava field photos, PlantVillage lab photos and PlantDoc real-scene photos (about 2 GB). Takes 15–30 minutes.

PlantDoc photos that duplicate each other or a lab photo are removed **before** splitting, so no photo is in both training and test. The line `plantdoc: N images kept, M duplicates dropped` reports this.

In [ ]:
%cd /content/acousticacre/ml
!python -m train.crop_prepare --out data/crop --max-per-class 1500

## 3. BEFORE: today's app model on the held-out field photos
This scores the crop pack that is in the app right now, with the app's own rules (including the wrong-crop check).

In [ ]:
!python -m eval.field_eval --pack /content/acousticacre/app/public/models/crop --data data/crop/test --source plantdoc_field --out before_plantdoc.json
!python -m eval.field_eval --pack /content/acousticacre/app/public/models/crop --data data/crop/test --source cassava_field --out before_cassava.json

## 4. Retrain with field photos added (GPU, 30–60 minutes)
Same model and settings as before; the only change is the added PlantDoc training photos.

In [ ]:
!python -m train.crop_train --data data/crop --out runs/crop --epochs 8 --quantize

## 5. AFTER: the new model on the same held-out photos

In [ ]:
!python -m eval.field_eval --pack runs/crop --data data/crop/test --source plantdoc_field --out after_plantdoc.json
!python -m eval.field_eval --pack runs/crop --data data/crop/test --source cassava_field --out after_cassava.json

## 6. Build the new pack and download everything
Send the four `*_plantdoc.json` / `*_cassava.json` files and `crop_report.json` back to Claude. **Only replace the app's model if the AFTER numbers are better** without making cassava worse.

In [ ]:
%cd /content/acousticacre
!python tools/pack_builder.py crop --run ml/runs/crop --out packs --version 0.2.0
!cp ml/runs/crop/report.json /content/crop_report.json
!cd ml && zip -q /content/field_results.zip before_plantdoc.json before_cassava.json after_plantdoc.json after_cassava.json
from google.colab import files
for f in ['/content/field_results.zip', '/content/crop_report.json', '/content/acousticacre/packs/crop-core-1-0.2.0.aapack']:
    files.download(f)